# Phase 4 -- Trial-count-matched SMI comparison (saline vs. sequential DCZ blocks)

Saline sessions are intentionally much shorter than their paired DCZ sessions (same-day design: short saline first, then a much longer DCZ session, to avoid exhausting the animal). Since both `calculate_SMI_improved`'s curve fit and `combined_reliability_test_improved`'s split-half correlation are trial-count-sensitive (more trials = more statistical power to pass either check, independent of any real change in tuning), a raw saline-vs-DCZ comparison at full trial count confounds "DCZ changed spatial coding" with "DCZ's session just has more trials." This notebook controls for that: split each DCZ session into sequential blocks matching saline's own trial count, and compare saline against each block separately -- which also surfaces whether the DREADD effect has an onset time-course across the DCZ session (block 1, closest in time to saline, vs. later blocks), something a single pooled comparison or random trial subsampling would wash out.

**Block construction:** `n_trials_target` = saline's own trial count. Blocks 1..k-1 are consecutive, non-overlapping tiles of the DCZ session (`[0:N)`, `[N:2N)`, ...); the LAST block is anchored to the *final* N trials instead of wherever consecutive tiling would leave it, so every block -- including the last -- is genuinely N-matched (the last block may overlap the second-to-last by a few trials if the DCZ trial count isn't an exact multiple of N; every other block is a clean, non-overlapping tile).

**Starts from `preproc.h5` directly** (`spatial_activity`/`norm_spatial_activity` -- both already spike-smoothed in `Preprocess.py`, see `smoothed_spatial_activity`; `calculate_SMI_improved` below applies its own additional spatial smoothing on top, `smoothing_sigma=1.0`, same as everywhere else in this pipeline) -- **not** from Phase 3's already-saved `*_smi_results_dreadd.h5`, since those are fixed at full trial count and this needs SMI *and* reliability recomputed per block.

**Reliability** is recomputed with `combined_reliability_test_improved` (not the simpler `test_cell_reliability`) -- the actual function `Preprocess.py` uses to produce `combined_reliable`, with the same V1 (non-RSC) parameters `Preprocess.py` itself uses: `cc_percentile=90, cohen_threshold=0.8, min_cc_threshold=0.1, min_pattern_corr=0.3, peak_distance_threshold=5, use_activity_threshold=True, activity_method='absolute_percentile'`. `helper/SMI_Calculation.py`'s `calculate_SMI_improved`, `helper/ReliabilityTesting.py`'s `combined_reliability_test_improved`, `helper/SMICalculation_LayerSpecific_SingleRecording.py`'s `filter_onset_response_cells`, and `helper/ResponseVisualization.py`'s `create_response_plot` are all reused completely unmodified throughout -- only the trial-axis slice of `spatial_activity` fed into them changes per block.

`n_shuffles=300` throughout (not the pipeline's usual 1000) since this is still exploratory -- also matches what `Preprocess.py` itself already defaults to for `combined_reliability_test_improved`. Bump back up if/when this becomes a settled, final analysis.

In [ ]:
import sys
sys.path.insert(0, r"C:\Users\jasmineyeo\Documents\GitHub\V1_SpatialModulation")

import os
import re
import glob
import json
from collections import Counter
from itertools import combinations

import numpy as np
import pandas as pd
import h5py
import matplotlib
matplotlib.use('Qt5Agg')  # for plt.show() popups
import matplotlib.pyplot as plt
from matplotlib import rcParams
from scipy.stats import kruskal, mannwhitneyu

rcParams['legend.fontsize'] = 20
rcParams['axes.labelsize'] = 20
rcParams['axes.titlesize'] = 25
rcParams['xtick.labelsize'] = 20
rcParams['ytick.labelsize'] = 20

# Existing pipeline code, reused via import -- not modified.
from helper import files
from helper.SMI_Calculation import calculate_SMI_improved
from helper.ReliabilityTesting import combined_reliability_test_improved
from helper.SMICalculation_LayerSpecific_SingleRecording import filter_onset_response_cells
from helper.ResponseVisualization import create_response_plot

N_SHUFFLES = 300  # exploratory -- bump back to 1000 once the approach is settled

# V1 (non-RSC) reliability parameters -- same branch Preprocess.py itself
# uses for this project's recordings (confirmed: this data is V1, not RSC).
RELIABILITY_KWARGS = dict(
    cc_percentile=90, cohen_threshold=0.8, min_cc_threshold=0.1,
    min_pattern_corr=0.3, peak_distance_threshold=5,
    use_activity_threshold=True, activity_method='absolute_percentile',
)

# Same defaults run_smi_analysis_session (3.SMICalculation.py) currently uses.
SMI_KWARGS = dict(
    exclude_first_bins=10, exclude_last_bins=10,
    segment_distance=28, exclude_start_cm=15, exclude_end_cm=10,
    smoothing_sigma=1.0,
)

# Point this at whichever animal you're processing -- kept for both so
# switching doesn't leave other TEST_* constants pointing at the wrong
# animal (same fix 4.SessionComparison.ipynb needed).
TEST_ANIMAL_DIR_JSY090 = r"D:\V1_SpatialModulation\2p\V1_prism_DREADD\JSY090_V1prism_DREADD"
TEST_ANIMAL_DIR_JSY093 = r"D:\V1_SpatialModulation\2p\V1_prism_DREADD\JSY093_V1prism_DREADD"
TEST_ANIMAL_DIR = TEST_ANIMAL_DIR_JSY090

## Setup -- `discover_smi_sessions`

Reused unchanged from every other notebook in this project -- needed just to locate each session's `tseries_dir` (so `preproc.h5` can be found) and `session_type` ('saline'/'dcz'/'baseline').

In [ ]:
def discover_smi_sessions(animal_dir):
    """
    Scan animal_dir for every already-computed *_smi_results_dreadd.h5
    file, labeling each by whichever known naming pattern its TSeries
    folder matches. Same disambiguation behavior as every other notebook
    here -- nothing is silently dropped on a label collision.

    Parameters
    ----------
    animal_dir : str

    Returns
    -------
    catalog : dict
        {label: {'save_path': str, 'session_type': str, 'tseries_dir': str}}
    """
    save_paths = sorted(glob.glob(os.path.join(animal_dir, '**', '*_smi_results_dreadd.h5'),
                                   recursive=True))

    entries = []  # (base_label, session_type, save_path, tseries_dir, tseries_name)
    unmatched = []

    for save_path in save_paths:
        tseries_dir = os.path.dirname(save_path)
        tseries_name = os.path.basename(tseries_dir)
        parent_dir = os.path.dirname(tseries_dir)
        parent_name = os.path.basename(parent_dir)

        upper_tseries = tseries_name.upper()

        if 'SAL' in upper_tseries:
            session_type = 'saline'
            base_label = f'{parent_name}_SALINE'
        elif 'DCZ' in upper_tseries:
            session_type = 'dcz'
            base_label = f'{parent_name}_DCZ'
        else:
            day_match = re.search(r'Day(\d+)', parent_name, re.IGNORECASE)
            if day_match:
                session_type = 'baseline'
                base_label = f'Day{day_match.group(1)}'
            else:
                session_type = 'unknown'
                base_label = tseries_name
                unmatched.append(base_label)

        entries.append((base_label, session_type, save_path, tseries_dir, tseries_name))

    label_counts = Counter(e[0] for e in entries)

    catalog = {}
    for base_label, session_type, save_path, tseries_dir, tseries_name in entries:
        label = f'{base_label}__{tseries_name}' if label_counts[base_label] > 1 else base_label

        if label in catalog:
            print(f"WARNING: label '{label}' still collides after disambiguation -- "
                  f"keeping {catalog[label]['save_path']}, skipping {save_path}")
            continue

        catalog[label] = {
            'save_path': save_path,
            'session_type': session_type,
            'tseries_dir': tseries_dir,
        }

    print(f"Discovered {len(catalog)} sessions with saved SMI results under {animal_dir}:")
    for label, info in catalog.items():
        print(f"  [{info['session_type']:>8}] {label}  <-  {info['save_path']}")

    collided_labels = [l for l, c in label_counts.items() if c > 1]
    if collided_labels:
        print(f"\n{len(collided_labels)} label(s) had multiple TSeries and were disambiguated: "
              f"{collided_labels}")

    if unmatched:
        print(f"\n{len(unmatched)} session(s) didn't match a known naming pattern "
              f"(labeled 'unknown'): {unmatched}")

    return catalog

## Setup -- `pair_saline_dcz_sessions`

Pairs each saline session with its same-day dcz session by shared *parent folder* (the `..._Saline_DCZ_N` folder both TSeries subfolders live under) -- matches on actual folder structure rather than label strings, so it's robust to whatever label `discover_smi_sessions` happened to assign, including a disambiguated `__TSeries...` suffix (exactly the kind of label mismatch that broke `4.SessionComparison.ipynb` twice already).

In [ ]:
def pair_saline_dcz_sessions(session_catalog):
    """
    Pair each saline session with its same-day dcz session, by shared
    parent folder -- robust to whatever label string discover_smi_sessions
    assigned, since it matches on the actual folder structure instead.

    Parameters
    ----------
    session_catalog : dict
        From discover_smi_sessions.

    Returns
    -------
    pairs : list of dict
        Each: {'group_name', 'saline_label', 'dcz_label'}. group_name is
        the shared parent folder's basename.
    """
    saline_entries = [(l, info) for l, info in session_catalog.items() if info['session_type'] == 'saline']
    dcz_entries = [(l, info) for l, info in session_catalog.items() if info['session_type'] == 'dcz']

    pairs = []
    for saline_label, saline_info in saline_entries:
        saline_parent = os.path.dirname(saline_info['tseries_dir'])
        matches = [(l, info) for l, info in dcz_entries
                   if os.path.dirname(info['tseries_dir']) == saline_parent]
        if len(matches) != 1:
            print(f"WARNING: saline session '{saline_label}' has {len(matches)} same-parent-folder dcz "
                  f"match(es) (expected 1) -- skipping. Parent: {saline_parent}")
            continue
        dcz_label, dcz_info = matches[0]
        pairs.append({
            'group_name': os.path.basename(saline_parent),
            'saline_label': saline_label,
            'dcz_label': dcz_label,
        })

    print(f"Paired {len(pairs)} saline/dcz session(s): {[p['group_name'] for p in pairs]}")
    return pairs

In [ ]:
# --- Try it on the real animal dir ---
smi_catalog = discover_smi_sessions(TEST_ANIMAL_DIR)
session_pairs = pair_saline_dcz_sessions(smi_catalog)
for p in session_pairs:
    print(p)

## Setup -- save utilities

Same generic helpers used throughout this project's other notebooks.

In [ ]:
def save_dataframe_csv(df, output_dir, filename):
    """
    Save a DataFrame to {output_dir}/{filename}, creating output_dir if
    needed.
    """
    os.makedirs(output_dir, exist_ok=True)
    save_path = os.path.join(output_dir, filename)
    df.to_csv(save_path, index=False)
    print(f"Saved -> {save_path}")
    return save_path


def save_figure_png(fig, output_dir, filename, dpi=150):
    """
    Save a matplotlib figure to {output_dir}/{filename}, creating
    output_dir if needed.
    """
    os.makedirs(output_dir, exist_ok=True)
    save_path = os.path.join(output_dir, filename)
    fig.savefig(save_path, dpi=dpi, bbox_inches='tight')
    print(f"Saved -> {save_path}")
    return save_path

## Setup -- layer assignment

`find_layer_curve_path`/`load_session_layer_cells_for_smi` reimplemented unchanged from `3.SMICalculation.py` (digit-prefixed module names aren't importable, same convention used throughout this pipeline) -- Phase 1's curve-based per-cell layer assignment. Layer identity is an anatomical/depth property of a cell, not derived from trials at all, so it's identical for every dcz block within a session -- loaded once per session here, not recomputed per block.

In [ ]:
def find_layer_curve_path(plane0_path, prefer_averaged=True):
    """
    Locate a session's Phase 1 layer-curve-results file, given its
    suite2p/plane0 path. Reimplemented unchanged from 3.SMICalculation.py.
    """
    tseries_dir = os.path.dirname(os.path.dirname(str(plane0_path)))
    averaged_matches = glob.glob(os.path.join(tseries_dir, '*_layer_curve_results_averaged.h5'))
    independent_matches = glob.glob(os.path.join(tseries_dir, '*_layer_curve_results.h5'))

    if prefer_averaged and averaged_matches:
        matches = averaged_matches
    elif independent_matches:
        matches = independent_matches
    elif averaged_matches:
        matches = averaged_matches
    else:
        raise FileNotFoundError(f"No *_layer_curve_results(_averaged).h5 found in {tseries_dir} "
                                 "-- has Phase 1 been run for this session?")

    if len(matches) > 1:
        print(f"WARNING: multiple matches in {tseries_dir}, using {matches[0]}")
    return matches[0]


def load_session_layer_cells_for_smi(plane0_path, layer_names=('L2/3', 'L4', 'L5', 'L6')):
    """
    Load Phase 1's curve-based layer assignment for one session, converted
    to {layer_name: indices}. Reimplemented unchanged from 3.SMICalculation.py.

    Parameters
    ----------
    plane0_path : str
    layer_names : tuple of str

    Returns
    -------
    layer_cells : dict
        {layer_name: numpy.ndarray of cell indices}.
    """
    layer_curve_path = find_layer_curve_path(plane0_path)

    with h5py.File(layer_curve_path, 'r') as f:
        layer_codes = f['layer_codes'][:]
        saved_layer_names = tuple(n.decode() if isinstance(n, bytes) else n
                                   for n in f['layer_names'][:])

    if saved_layer_names != layer_names:
        print(f"NOTE: saved layer_names {saved_layer_names} differ in order from "
              f"the requested {layer_names} -- using the saved order.")

    layer_cells = {
        name: np.where(layer_codes == code)[0]
        for code, name in enumerate(saved_layer_names)
    }

    print(f"Loaded layer assignment from {layer_curve_path}")
    for name, idx in layer_cells.items():
        print(f"  {name}: {len(idx)} cells")

    return layer_cells


def load_session_layer_of_cell(tseries_dir, n_cells):
    """
    Per-cell layer label array for one session -- wraps
    load_session_layer_cells_for_smi (Phase 1's output) into the shape
    build_trial_matched_comparison_table needs.

    Parameters
    ----------
    tseries_dir : str
    n_cells : int
        Total cells in this session (spatial_activity.shape[0]).

    Returns
    -------
    layer_of_cell : numpy.ndarray of object, shape (n_cells,)
        layer_of_cell[i] is 'L2/3'/'L4'/'L5'/'L6', or None if cell i isn't
        assigned to any of the four layers.
    """
    plane0_path = os.path.join(tseries_dir, 'suite2p', 'plane0')
    layer_cells = load_session_layer_cells_for_smi(plane0_path)

    layer_of_cell = np.full(n_cells, None, dtype=object)
    for layer_name, idx in layer_cells.items():
        layer_of_cell[idx] = layer_name
    return layer_of_cell

In [ ]:
# --- Try it on the real first saline/dcz sessions ---
test_saline_layer_of_cell = load_session_layer_of_cell(
    smi_catalog[test_pair['saline_label']]['tseries_dir'], test_saline_activity.shape[0]
)
test_dcz_layer_of_cell = load_session_layer_of_cell(
    smi_catalog[test_pair['dcz_label']]['tseries_dir'], test_dcz_activity.shape[0]
)

## Function -- `load_session_spatial_data`

Pulls `spatial_activity`/`norm_spatial_activity`/`bin_centers` straight from one session's `preproc.h5` -- the starting point for everything in this notebook.

In [ ]:
def load_session_spatial_data(tseries_dir):
    """
    Pull spatial_activity/norm_spatial_activity/bin_centers straight from
    one session's preproc.h5.

    Parameters
    ----------
    tseries_dir : str

    Returns
    -------
    spatial_activity : numpy.ndarray
        (n_cells, n_trials, n_bins) -- Preprocess.py's own spike-smoothed
        version (SpikeSmoothing.spatial_smooth's output), the same array
        calculate_SMI_improved/combined_reliability_test_improved expect.
    norm_spatial_activity : numpy.ndarray
        Same shape, per-lap min/max-normalized (RT.normalize_spatial_activity).
    bin_centers : numpy.ndarray
        (n_bins,) raw cm-scale bin centers.
    """
    preproc_files = glob.glob(os.path.join(tseries_dir, "*preproc*.h5"))
    if not preproc_files:
        raise FileNotFoundError(f"No *preproc*.h5 found in {tseries_dir}")
    preproc_data = files.read_h5(preproc_files[0])
    spatial_activity = preproc_data['spatial_activity']
    norm_spatial_activity = preproc_data['norm_spatial_activity']
    bin_centers = preproc_data['bin_centers']
    print(f"Loaded spatial_activity {spatial_activity.shape} from {tseries_dir}")
    return spatial_activity, norm_spatial_activity, bin_centers

In [ ]:
# --- Try it on the real first saline session ---
test_pair = session_pairs[0]
test_saline_activity, test_saline_norm, test_saline_bins = load_session_spatial_data(
    smi_catalog[test_pair['saline_label']]['tseries_dir']
)
test_dcz_activity, test_dcz_norm, test_dcz_bins = load_session_spatial_data(
    smi_catalog[test_pair['dcz_label']]['tseries_dir']
)
print(f"\nsaline trials: {test_saline_activity.shape[1]}   dcz trials: {test_dcz_activity.shape[1]}")

## Function -- `split_into_matched_blocks`

Implements the block-construction rule from the markdown above: consecutive non-overlapping tiles of size N, except the last block is anchored to the final N trials so every block -- including the last -- is genuinely N-matched.

In [ ]:
def split_into_matched_blocks(n_trials_source, n_trials_target):
    """
    Split a session with n_trials_source trials into sequential blocks of
    exactly n_trials_target trials each: blocks 0..k-2 are consecutive,
    non-overlapping [i*N:(i+1)*N); the LAST block is anchored to the final
    N trials (source[-N:]) instead of wherever consecutive tiling would
    leave it, so every block -- including the last -- is genuinely
    N-matched (the last block may overlap the second-to-last by a few
    trials if n_trials_source isn't an exact multiple of N; every other
    block is a clean, non-overlapping tile).

    Parameters
    ----------
    n_trials_source : int
        Total trials in the longer session (e.g. dcz).
    n_trials_target : int
        Trials per block -- the shorter session's own trial count (e.g.
        saline).

    Returns
    -------
    blocks : list of (int, int)
        (start, end) trial-index ranges, in original temporal order --
        block 0 is closest in time to the shorter session, last block is
        furthest.
    """
    if n_trials_target <= 0:
        raise ValueError(f"n_trials_target must be > 0, got {n_trials_target}")
    if n_trials_target > n_trials_source:
        raise ValueError(f"n_trials_target ({n_trials_target}) > n_trials_source ({n_trials_source}) "
                          f"-- can't carve a block this big out of the source session.")

    n_full_blocks = n_trials_source // n_trials_target
    remainder = n_trials_source % n_trials_target

    blocks = [(i * n_trials_target, (i + 1) * n_trials_target) for i in range(n_full_blocks)]
    if remainder > 0:
        blocks.append((n_trials_source - n_trials_target, n_trials_source))

    print(f"Split {n_trials_source} trials into {len(blocks)} block(s) of {n_trials_target}: {blocks}")
    return blocks

In [ ]:
# --- Sanity check on made-up numbers first (65 dcz trials, 20 saline trials) ---
example_blocks = split_into_matched_blocks(65, 20)
assert example_blocks == [(0, 20), (20, 40), (40, 60), (45, 65)], example_blocks
print("OK -- matches expected option-3 behavior (last block anchored to the final 20 trials).")

# --- Now the real DCZ1 pair ---
test_blocks = split_into_matched_blocks(test_dcz_activity.shape[1], test_saline_activity.shape[1])

## Function -- `run_smi_and_reliability_for_block`

The core per-block computation: reliability (`combined_reliability_test_improved`, V1 parameters) -> onset/reward filtering (`filter_onset_response_cells`) -> SMI (`calculate_SMI_improved`) -- mirrors `run_smi_analysis_session`'s own processing order (`3.SMICalculation.py`) exactly, except reliability is recomputed fresh on this block's own trial-slice instead of read from `preproc.h5`'s already-saved (full-session) `combined_reliable`. Every function called here is reused completely unmodified.

In [ ]:
def run_smi_and_reliability_for_block(spatial_activity, bin_centers, block_range,
                                       n_shuffles=N_SHUFFLES,
                                       reliability_kwargs=RELIABILITY_KWARGS,
                                       smi_kwargs=SMI_KWARGS):
    """
    Recompute SMI + reliability from scratch on one trial-index block of
    spatial_activity.

    Parameters
    ----------
    spatial_activity : numpy.ndarray
        (n_cells, n_trials, n_bins) -- the FULL session's array; block_range
        indexes into its trial axis.
    bin_centers : numpy.ndarray
        Raw cm-scale bin centers for this session.
    block_range : (int, int)
        (start, end) trial-index range from split_into_matched_blocks (or
        (0, n_trials) for an unblocked/whole session like saline).
    n_shuffles : int
        Passed to combined_reliability_test_improved.
    reliability_kwargs : dict
        Passed to combined_reliability_test_improved (V1/non-RSC defaults).
    smi_kwargs : dict
        exclude_first_bins/exclude_last_bins go to filter_onset_response_cells;
        segment_distance/exclude_start_cm/exclude_end_cm/smoothing_sigma go
        to calculate_SMI_improved.

    Returns
    -------
    result : dict
        {'n_trials', 'block_range', 'combined_reliable', 'pattern_reliable',
        'active_cells', 'analysis_reliable_cells', 'valid_cells_mask',
        'SMI', 'avg_cc', 'cohen_d'}.
    """
    start, end = block_range
    block_activity = spatial_activity[:, start:end, :]
    n_trials = block_activity.shape[1]

    # Same bin-center rescaling run_smi_analysis_session uses -- segment_distance/
    # exclude_start_cm/exclude_end_cm are calibrated in that rescaled space.
    shifted_centers = bin_centers - np.min(bin_centers)
    scaled_bin_centers = shifted_centers * (np.size(bin_centers) / np.max(shifted_centers))

    (combined_reliable, reliable_cells, pattern_reliable, avg_cc, cohens_d,
     odd_even_corr, peak_distances, active_cells) = combined_reliability_test_improved(
        block_activity, n_shuffles=n_shuffles, **reliability_kwargs
    )

    non_onset_cells, rejected_info = filter_onset_response_cells(
        block_activity, scaled_bin_centers, combined_reliable,
        exclude_first_bins=smi_kwargs['exclude_first_bins'],
        exclude_last_bins=smi_kwargs['exclude_last_bins'],
    )
    analysis_reliable_cells = combined_reliable & non_onset_cells

    smi_results = calculate_SMI_improved(
        block_activity, scaled_bin_centers, analysis_reliable_cells,
        segment_distance=smi_kwargs['segment_distance'],
        exclude_start_cm=smi_kwargs['exclude_start_cm'],
        exclude_end_cm=smi_kwargs['exclude_end_cm'],
        smoothing_sigma=smi_kwargs['smoothing_sigma'],
    )

    print(f"  Block [{start}:{end}] ({n_trials} trials): "
          f"combined_reliable={int(combined_reliable.sum())}, "
          f"analysis_reliable={int(analysis_reliable_cells.sum())}, "
          f"valid={int(smi_results['reliable_valid_cells'].sum())}")

    return {
        'n_trials': n_trials,
        'block_range': block_range,
        'combined_reliable': combined_reliable,
        'pattern_reliable': pattern_reliable,
        'active_cells': active_cells,
        'analysis_reliable_cells': analysis_reliable_cells,
        'valid_cells_mask': smi_results['reliable_valid_cells'],
        'SMI': smi_results['SMI'],
        'avg_cc': avg_cc,
        'cohen_d': cohens_d,
    }

In [ ]:
# --- Try it on the real saline session (single implicit block = whole session) ---
test_saline_result = run_smi_and_reliability_for_block(
    test_saline_activity, test_saline_bins, (0, test_saline_activity.shape[1])
)

## Function -- `run_trial_matched_comparison`

Loads one saline/dcz pair, splits dcz into matched blocks, and runs `run_smi_and_reliability_for_block` on saline (as one implicit whole-session block) plus every dcz block -- `'saline'`, `'dcz_block_1'`, `'dcz_block_2'`, ... in temporal order.

In [ ]:
def run_trial_matched_comparison(session_catalog, saline_label, dcz_label, n_shuffles=N_SHUFFLES):
    """
    Loads saline (implicit single 'block' = full session) and dcz (split
    into matched blocks via split_into_matched_blocks), and runs
    run_smi_and_reliability_for_block on each.

    Parameters
    ----------
    session_catalog : dict
        From discover_smi_sessions.
    saline_label, dcz_label : str
    n_shuffles : int

    Returns
    -------
    block_results : dict
        {condition_name: run_smi_and_reliability_for_block(...) result} --
        condition_name is 'saline', 'dcz_block_1', 'dcz_block_2', ... in
        temporal order (dict insertion order).
    session_data : dict
        {condition_name: {'spatial_activity', 'norm_spatial_activity',
        'bin_centers', 'layer_of_cell'}} -- the block-sliced arrays
        actually used, so the response-plot/layer-table steps don't need
        to reload/reslice anything. layer_of_cell is identical across all
        of a session's blocks (same FOV, same cells) -- loaded once, not
        recomputed per block.
    """
    saline_info = session_catalog[saline_label]
    dcz_info = session_catalog[dcz_label]

    saline_activity, saline_norm, saline_bins = load_session_spatial_data(saline_info['tseries_dir'])
    dcz_activity, dcz_norm, dcz_bins = load_session_spatial_data(dcz_info['tseries_dir'])

    saline_layer_of_cell = load_session_layer_of_cell(saline_info['tseries_dir'], saline_activity.shape[0])
    dcz_layer_of_cell = load_session_layer_of_cell(dcz_info['tseries_dir'], dcz_activity.shape[0])

    n_trials_saline = saline_activity.shape[1]
    n_trials_dcz = dcz_activity.shape[1]
    blocks = split_into_matched_blocks(n_trials_dcz, n_trials_saline)

    block_results = {}
    session_data = {}

    print(f"\n--- saline ({saline_label}) ---")
    block_results['saline'] = run_smi_and_reliability_for_block(
        saline_activity, saline_bins, (0, n_trials_saline), n_shuffles=n_shuffles
    )
    session_data['saline'] = {
        'spatial_activity': saline_activity,
        'norm_spatial_activity': saline_norm,
        'bin_centers': saline_bins,
        'layer_of_cell': saline_layer_of_cell,
    }

    print(f"\n--- dcz ({dcz_label}), {len(blocks)} block(s) ---")
    for i, block_range in enumerate(blocks, start=1):
        condition_name = f'dcz_block_{i}'
        block_results[condition_name] = run_smi_and_reliability_for_block(
            dcz_activity, dcz_bins, block_range, n_shuffles=n_shuffles
        )
        start, end = block_range
        session_data[condition_name] = {
            'spatial_activity': dcz_activity[:, start:end, :],
            'norm_spatial_activity': dcz_norm[:, start:end, :],
            'bin_centers': dcz_bins,
            'layer_of_cell': dcz_layer_of_cell,  # same for every dcz block -- same session/FOV
        }

    return block_results, session_data

In [ ]:
# --- Try it on the real DCZ1 pair (or whichever session_pairs[0] resolved to) ---
test_block_results, test_session_data = run_trial_matched_comparison(
    smi_catalog, test_pair['saline_label'], test_pair['dcz_label']
)
print(f"\nConditions: {list(test_block_results.keys())}")

## Function -- `build_trial_matched_comparison_table`

Per-cell-row table -- `cell_idx, SMI, valid, analysis_reliable, layer, condition` -- one row per cell per condition (saline, dcz_block_1, ..., dcz_block_n). Same shape as Phase 4's `build_comparison_table` output, so a Phase-5-style layer analysis can consume it directly (just with more `condition` categories than the usual baseline/saline/dcz).

In [ ]:
def build_trial_matched_comparison_table(block_results, session_data):
    """
    Build a per-cell-row table -- cell_idx, SMI, valid, analysis_reliable,
    layer, condition -- one row per cell per condition.

    Parameters
    ----------
    block_results : dict
        From run_trial_matched_comparison.
    session_data : dict
        From run_trial_matched_comparison (needs 'layer_of_cell' per
        condition).

    Returns
    -------
    df : pandas.DataFrame
    """
    condition_dfs = []
    for condition_name, result in block_results.items():
        n_cells = len(result['SMI'])
        layer_of_cell = session_data[condition_name]['layer_of_cell']
        condition_dfs.append(pd.DataFrame({
            'cell_idx': np.arange(n_cells),
            'SMI': result['SMI'],
            'valid': result['valid_cells_mask'],
            'analysis_reliable': result['analysis_reliable_cells'],
            'layer': layer_of_cell,
            'condition': condition_name,
        }))

    df = pd.concat(condition_dfs, ignore_index=True)
    print(f"Built trial-matched comparison table: {len(df)} cell-rows across {len(block_results)} condition(s)")
    return df

In [ ]:
# --- Try it on the real DCZ1 block_results/session_data ---
test_comparison_table = build_trial_matched_comparison_table(test_block_results, test_session_data)
print(test_comparison_table.groupby('condition')['valid'].agg(['sum', 'count']))

## Function -- `plot_smi_across_blocks`

Single violin+strip plot of SMI (`valid_cells_mask`-filtered), saline vs. every dcz block, in temporal order -- same visual style as `4.SessionComparison.ipynb`'s `plot_smi_comparison`, reimplemented here for conditions = these blocks.

In [ ]:
def plot_smi_across_blocks(block_results, title=''):
    """
    Single violin+strip plot of SMI (valid_cells_mask-filtered) across
    saline, dcz_block_1, ..., dcz_block_n, in temporal order.

    Parameters
    ----------
    block_results : dict
        From run_trial_matched_comparison.
    title : str

    Returns
    -------
    fig : matplotlib.figure.Figure
    summary_df : pandas.DataFrame
        One row per condition: n, median_SMI, mean_SMI.
    """
    condition_order = list(block_results.keys())  # already temporal (dict insertion order)

    n_dcz_blocks = len(condition_order) - 1
    dcz_colors = plt.cm.Purples(np.linspace(0.4, 0.9, max(n_dcz_blocks, 1)))
    color_map = {'saline': 'tab:orange'}
    for i, cond in enumerate(condition_order[1:]):
        color_map[cond] = dcz_colors[i]

    data_by_condition = []
    summary_rows = []
    for cond in condition_order:
        result = block_results[cond]
        vals = result['SMI'][result['valid_cells_mask']]
        data_by_condition.append(vals)
        summary_rows.append({
            'condition': cond,
            'n': len(vals),
            'median_SMI': float(np.median(vals)) if len(vals) else np.nan,
            'mean_SMI': float(np.mean(vals)) if len(vals) else np.nan,
        })
    summary_df = pd.DataFrame(summary_rows)

    fig, ax = plt.subplots(figsize=(2.5 * len(condition_order) + 2, 8))
    nonempty = [(i, d) for i, d in enumerate(data_by_condition) if len(d) > 0]
    if nonempty:
        parts = ax.violinplot([d for _, d in nonempty], positions=[i + 1 for i, _ in nonempty],
                               showmedians=True)
        for (i, _), body in zip(nonempty, parts['bodies']):
            body.set_facecolor(color_map[condition_order[i]])
            body.set_alpha(0.4)

    rng = np.random.default_rng(0)
    for i, vals in enumerate(data_by_condition):
        if len(vals) == 0:
            continue
        jitter = rng.uniform(-0.08, 0.08, size=len(vals))
        ax.scatter(np.full(len(vals), i + 1) + jitter, vals,
                   color=color_map[condition_order[i]], s=15, alpha=0.5)

    ax.set_xticks(range(1, len(condition_order) + 1))
    ax.set_xticklabels([f"{c}\n(n={r['n']})" for c, r in zip(condition_order, summary_rows)])
    ax.set_ylabel('SMI')
    ax.set_title(title)
    ax.axhline(0, color='gray', linestyle='--', alpha=0.5)

    plt.tight_layout()
    return fig, summary_df

In [ ]:
# --- Try it on the real DCZ1 block_results ---
test_smi_fig, test_summary_df = plot_smi_across_blocks(
    test_block_results, title=f"{test_pair['group_name']} -- trial-matched SMI"
)
print(test_summary_df.to_string(index=False))
# Not calling plt.show() here -- see the markdown above cell 27's functions
# for why. Uncomment to look at this one figure interactively:
# plt.show()

## Function -- `plot_response_plots_across_blocks`

`create_response_plot` (unmodified), one figure per (mask tier, condition) -- same three-tier split (`analysis_reliable_cells` / `valid_cells_mask` / reliable-rejected-from-valid) already used in `4.SessionComparison.ipynb`'s response-plot diagnostic, just across saline + every dcz block instead of 2 sessions.

In [ ]:
def plot_response_plots_across_blocks(block_results, session_data):
    """
    create_response_plot for saline, dcz_block_1, ..., dcz_block_n --
    three mask tiers each (analysis_reliable_cells / valid_cells_mask /
    reliable-rejected-from-valid). Figures are created and returned but
    NOT displayed (no plt.show()) -- call plt.show() yourself on a
    returned fig if you want to look at one interactively; save_trial_
    matched_outputs saves (and closes) every one regardless.

    Parameters
    ----------
    block_results : dict
        From run_trial_matched_comparison.
    session_data : dict
        From run_trial_matched_comparison.

    Returns
    -------
    figs : dict
        {(mask_name, condition_name): fig}.
    """
    mask_tiers = ['analysis_reliable_cells', 'valid_cells_mask', 'rejected_from_valid']
    figs = {}
    for condition_name, result in block_results.items():
        norm_activity = session_data[condition_name]['norm_spatial_activity']
        masks = {
            'analysis_reliable_cells': result['analysis_reliable_cells'],
            'valid_cells_mask': result['valid_cells_mask'],
            'rejected_from_valid': result['analysis_reliable_cells'] & ~result['valid_cells_mask'],
        }
        for mask_name in mask_tiers:
            mask = masks[mask_name]
            if not mask.any():
                print(f"  {condition_name} / {mask_name}: 0 cells -- skipping (nothing to plot).")
                continue
            fig, _ = create_response_plot(norm_activity, mask, clim=(0, 1))
            fig.suptitle(f"{condition_name}\n{mask_name} (n={int(mask.sum())})", fontsize=14)
            figs[(mask_name, condition_name)] = fig

    return figs

In [ ]:
# --- Try it on the real DCZ1 block_results/session_data ---
test_response_figs = plot_response_plots_across_blocks(test_block_results, test_session_data)

## Function -- save everything + run it across every saline/dcz pair, for every animal

`save_trial_matched_outputs` saves one group's SMI summary table, SMI comparison figure, and every response-plot figure -- and closes each figure right after saving it (`plt.close`), so a long unattended run doesn't accumulate hundreds of open figures in memory. `run_all_trial_matched_comparisons` loops the whole pipeline (`run_trial_matched_comparison` -> `plot_smi_across_blocks` -> `plot_response_plots_across_blocks` -> save) over every pair from `pair_saline_dcz_sessions`.

**No plot windows pop up anywhere in this notebook's driver path anymore** -- every figure is created, saved to disk, and closed without ever calling `plt.show()`, so the whole thing (including the dual-animal loop below) can run start to finish unattended. Review results afterward from the saved PNGs, or call `plt.show()` yourself on any returned `fig` object if you want to look at one interactively.

In [ ]:
def save_trial_matched_outputs(output_dir, group_name, summary_df, smi_fig, response_figs, comparison_table=None):
    """
    Save one group's trial-matched outputs: SMI summary table, SMI
    comparison figure, every response-plot figure, and (if given) the
    per-cell comparison table (cell_idx/SMI/valid/analysis_reliable/layer/
    condition) that a Phase-5-style layer analysis needs.

    Parameters
    ----------
    output_dir : str
    group_name : str
    summary_df : pandas.DataFrame
        From plot_smi_across_blocks.
    smi_fig : matplotlib.figure.Figure
        From plot_smi_across_blocks.
    response_figs : dict
        From plot_response_plots_across_blocks.
    comparison_table : pandas.DataFrame, optional
        From build_trial_matched_comparison_table. Saved as
        '{group_name}_trial_matched_comparison_table.csv' if given.

    Returns
    -------
    saved_paths : dict
    """
    saved_paths = {
        'summary': save_dataframe_csv(summary_df, output_dir, f"{group_name}_trial_matched_smi_summary.csv"),
        'smi_plot': save_figure_png(smi_fig, output_dir, f"{group_name}_trial_matched_smi_comparison.png"),
    }
    if comparison_table is not None:
        saved_paths['comparison_table'] = save_dataframe_csv(
            comparison_table, output_dir, f"{group_name}_trial_matched_comparison_table.csv")
    plt.close(smi_fig)

    response_dir = os.path.join(output_dir, f"{group_name}_response_plots")
    for (mask_name, condition_name), fig in response_figs.items():
        key = f"{condition_name}_{mask_name}"
        saved_paths[key] = save_figure_png(fig, response_dir, f"{key}.png")
        plt.close(fig)

    return saved_paths


def run_all_trial_matched_comparisons(session_catalog, pairs, output_dir, n_shuffles=N_SHUFFLES):
    """
    Loop the whole trial-matched pipeline over every saline/dcz pair.

    Parameters
    ----------
    session_catalog : dict
    pairs : list of dict
        From pair_saline_dcz_sessions.
    output_dir : str
    n_shuffles : int

    Returns
    -------
    results_by_group : dict
        {group_name: {'block_results', 'session_data', 'summary_df', 'saved_paths'}}.
    """
    results_by_group = {}
    failed_groups = {}
    for pair in pairs:
        group_name = pair['group_name']
        print(f"\n{'='*90}\nTRIAL-MATCHED -- {group_name}\n{'='*90}")

        try:
            block_results, session_data = run_trial_matched_comparison(
                session_catalog, pair['saline_label'], pair['dcz_label'], n_shuffles=n_shuffles
            )

            smi_fig, summary_df = plot_smi_across_blocks(block_results, title=f"{group_name} -- trial-matched SMI")
            print(summary_df.to_string(index=False))

            response_figs = plot_response_plots_across_blocks(block_results, session_data)
            comparison_table = build_trial_matched_comparison_table(block_results, session_data)

            saved_paths = save_trial_matched_outputs(output_dir, group_name, summary_df, smi_fig, response_figs,
                                                       comparison_table=comparison_table)

            results_by_group[group_name] = {
                'block_results': block_results,
                'session_data': session_data,
                'summary_df': summary_df,
                'comparison_table': comparison_table,
                'saved_paths': saved_paths,
            }
        except Exception as exc:
            print(f"FAILED on group '{group_name}' -- {type(exc).__name__}: {exc}")
            print(f"  Skipping this group and continuing with the rest. "
                  f"(Any groups already completed above are already saved to disk.)")
            failed_groups[group_name] = exc
            continue

    if failed_groups:
        print(f"\n{len(failed_groups)}/{len(pairs)} group(s) failed and were skipped: "
              f"{list(failed_groups.keys())}")

    return results_by_group

In [ ]:
# --- Run it across every saline/dcz pair, for both animals ---
ANIMAL_CONFIGS = [
    {'animal_label': 'JSY090', 'animal_dir': TEST_ANIMAL_DIR_JSY090},
    {'animal_label': 'JSY093', 'animal_dir': TEST_ANIMAL_DIR_JSY093},
]

trial_matched_results_by_animal = {}
for animal_cfg in ANIMAL_CONFIGS:
    print(f"\n{'#'*90}\n{animal_cfg['animal_label']}\n{'#'*90}")
    animal_catalog = discover_smi_sessions(animal_cfg['animal_dir'])
    animal_pairs = pair_saline_dcz_sessions(animal_catalog)
    animal_output_dir = os.path.join(animal_cfg['animal_dir'], 'Phase4_TrialMatched_Results')
    trial_matched_results_by_animal[animal_cfg['animal_label']] = run_all_trial_matched_comparisons(
        animal_catalog, animal_pairs, animal_output_dir
    )